# Factor Transport Decomposition

This notebook demonstrates the factor decomposition of optimal transport maps and transport-space PCA on synthetic factor-driven data.

**Key idea:** If returns follow $r_i = \sum_k \beta_{ik} f_k + \varepsilon_i$, then distributional shifts between dates are largely driven by factor returns.  We can decompose the OT displacement field onto the factor-loading structure to attribute each shift to specific factors.


In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.synthetic import generate_factor_driven_panel
from otreturns.distributions import DistributionPanel
from otreturns.factors import (
    transport_factor_decomposition,
    rolling_factor_attribution,
    variance_decomposition_ot,
    transport_pca,
)
from otreturns.transport import optimal_transport_map_1d, transport_map_displacement
print("Imports OK")


Imports OK


## 1. Generate Factor-Driven Panel

In [2]:
n_dates, n_stocks, n_factors = 80, 500, 3
panel_df, factor_returns, factor_loadings = generate_factor_driven_panel(
    n_dates=n_dates, n_stocks=n_stocks, n_factors=n_factors, seed=0
)
panel = DistributionPanel.from_panel(panel_df, min_stocks=50, winsorize=0.005)
dates = panel.dates
T = len(dates)
print(f"Panel: {T} dates, {panel[dates[0]].n} stocks")
print(f"Factor returns shape: {factor_returns.shape}")
print(f"Factor loadings shape: {factor_loadings.shape}")

# Factor vol regime: doubles at midpoint
midpoint = T // 2
print(f"\nFactor vols (first half): {factor_returns[:midpoint].std(axis=0).round(5)}")
print(f"Factor vols (second half): {factor_returns[midpoint:].std(axis=0).round(5)}  (2x higher)")


Panel: 80 dates, 500 stocks
Factor returns shape: (80, 3)
Factor loadings shape: (500, 3)

Factor vols (first half): [0.00959 0.00832 0.00453]
Factor vols (second half): [0.02163 0.01437 0.01149]  (2x higher)


## 2. Single-Step Transport Factor Decomposition

We decompose the displacement field $d(u) = F_{\nu}^{-1}(u) - F_{\mu}^{-1}(u)$ onto factor-loading directions $\bar{\beta}(u)$ via OLS.

- $\bar{\beta}_k(u)$ = average loading of factor $k$ for stocks near quantile $u$.
- OLS gives $\hat{\alpha}_k$ = intensity of factor $k$'s contribution.
- $R^2$ measures how much the displacement is factor-driven.


In [3]:
factor_names = [f"f{k}" for k in range(n_factors)]
loadings_df  = pd.DataFrame(factor_loadings[:n_stocks], columns=factor_names)

# Compare a low-vol and a high-vol consecutive pair
t_low  = midpoint - 2
t_high = midpoint + 2

source_low  = panel[dates[t_low]]
target_low  = panel[dates[t_low + 1]]
source_high = panel[dates[t_high]]
target_high = panel[dates[t_high + 1]]

decomp_low  = transport_factor_decomposition(source_low,  target_low,  loadings_df.values, factor_names)
decomp_high = transport_factor_decomposition(source_high, target_high, loadings_df.values, factor_names)

print(f"Low-vol period (t={t_low}):")
print(f"  R²             = {decomp_low['r_squared']:.4f}")
print(f"  factor_fraction= {decomp_low['factor_fraction']:.4f}")
print(f"High-vol period (t={t_high}):")
print(f"  R²             = {decomp_high['r_squared']:.4f}")
print(f"  factor_fraction= {decomp_high['factor_fraction']:.4f}")


Low-vol period (t=38):
  R²             = 0.8807
  factor_fraction= 0.8807
High-vol period (t=42):
  R²             = 0.9691
  factor_fraction= 0.9691


In [4]:
u = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
palette = ["steelblue", "darkorange", "forestgreen"]

for row_idx, (decomp, title, src, tgt) in enumerate([
        (decomp_low,  f"Low-vol (t={t_low})",  source_low,  target_low),
        (decomp_high, f"High-vol (t={t_high})", source_high, target_high),
]):
    ax_d = axes[row_idx, 0]
    ax_f = axes[row_idx, 1]

    d_field = decomp["displacement_field"]
    ax_d.plot(u, d_field, color="black", lw=2, label="Total displacement d(u)")
    for k, name in enumerate(factor_names):
        ax_d.plot(u, decomp["factor_displacement_fields"][name],
                  color=palette[k], lw=1.5, ls="--", label=f"{name} component")
    ax_d.axhline(0, color="gray", lw=0.5)
    ax_d.set_xlabel("Quantile u"); ax_d.set_ylabel("Displacement")
    ax_d.set_title(f"Displacement field — {title}  (R²={decomp['r_squared']:.3f})")
    ax_d.legend(fontsize=8); ax_d.grid(alpha=0.3)

    contribs = [decomp["factor_contributions"][n] for n in factor_names]
    ax_f.bar(factor_names, contribs, color=palette[:n_factors], alpha=0.8)
    ax_f.axhline(0, color="black", lw=0.8)
    ax_f.set_xlabel("Factor"); ax_f.set_ylabel("α (contribution)")
    ax_f.set_title(f"Factor contributions — {title}")
    ax_f.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../figures/nb04_decomposition.png", dpi=100)
plt.show()


## 3. Rolling Factor Attribution

For each consecutive date pair $(t-1, t)$ we run the decomposition and track how much of the distributional shift is factor-explained over time.

The factor explained fraction is expected to increase in the high-vol regime (midpoint onwards) because factor shocks are larger relative to idiosyncratic noise.


In [5]:
df_attr = rolling_factor_attribution(panel, loadings_df, window=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

t_idx = range(len(df_attr))

axes[0].plot(t_idx, df_attr["total_shift_W2"], color="steelblue", lw=1.5)
axes[0].axvline(midpoint, color="red", ls="--", alpha=0.6, label="Vol regime change")
axes[0].set_ylabel("W₂ shift"); axes[0].legend(fontsize=9)
axes[0].set_title("Total distributional shift W₂(μₜ₋₁, μₜ)"); axes[0].grid(alpha=0.3)

axes[1].plot(t_idx, df_attr["factor_r_squared"], color="darkorange", lw=1.5)
axes[1].axhline(0.5, color="gray", ls=":", alpha=0.7)
axes[1].axvline(midpoint, color="red", ls="--", alpha=0.6)
axes[1].set_ylim(0, 1); axes[1].set_ylabel("Factor R²")
axes[1].set_title("Factor-explained fraction of distributional shift"); axes[1].grid(alpha=0.3)

for k, (name, col) in enumerate(zip(factor_names, palette)):
    axes[2].plot(t_idx, df_attr[f"{name}_contribution"], color=col, lw=1.3, label=name)
axes[2].axhline(0, color="black", lw=0.5)
axes[2].axvline(midpoint, color="red", ls="--", alpha=0.6)
axes[2].set_xlabel("Date"); axes[2].set_ylabel("α_k contribution")
axes[2].set_title("Per-factor contributions"); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/nb04_rolling_attribution.png", dpi=100)
plt.show()
print(f"Mean R² (low vol)  : {df_attr[df_attr.index<midpoint-1]['factor_r_squared'].mean():.4f}")
print(f"Mean R² (high vol) : {df_attr[df_attr.index>=midpoint-1]['factor_r_squared'].mean():.4f}")


Mean R² (low vol)  : 0.4925
Mean R² (high vol) : 0.6841


## 4. Transport PCA

**Bigot, Cazelles & Papadakis (2017)** propose PCA in the tangent space of the Wasserstein space.

Given the Wasserstein barycenter $\bar{\mu}$, the **log-map** of $\mu_t$ at $\bar{\mu}$ is the tangent vector:
$$v_t(u) = F_{\mu_t}^{-1}(u) - F_{\bar{\mu}}^{-1}(u)$$

Standard PCA on the matrix $V \in \mathbb{R}^{T \times M}$ finds the dominant directions of distributional variation.

Typical interpretations:
- **PC1**: mean shift (bull/bear)
- **PC2**: dispersion change (vol regime)
- **PC3**: skewness/tail change


In [6]:
pca_result = transport_pca(panel, n_components=3, n_support=150)
evr    = pca_result["explained_variance_ratio"]
scores = pca_result["scores"]

print("Transport PCA — explained variance ratio:")
for i, v in enumerate(evr):
    print(f"  PC{i+1}: {v:.4f}  ({v*100:.1f}%)")
print(f"  Cumulative: {evr.cumsum()[-1]*100:.1f}%")

u_pca = np.linspace(0.01, 0.99, 150)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Component directions in tangent space
for i, (comp, col) in enumerate(zip(pca_result["components"], palette)):
    axes[0].plot(u_pca, comp, color=col, lw=2,
                 label=f"PC{i+1}  ({evr[i]*100:.1f}%)")
axes[0].axhline(0, color="black", lw=0.5, ls="--")
axes[0].set_xlabel("Quantile u"); axes[0].set_ylabel("Tangent direction")
axes[0].set_title("PCA components in tangent space")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Scores PC1 vs PC2 coloured by time
sc = axes[1].scatter(scores[:, 0], scores[:, 1], c=range(T), cmap="plasma", s=20)
axes[1].axvline(0, color="gray", lw=0.5); axes[1].axhline(0, color="gray", lw=0.5)
plt.colorbar(sc, ax=axes[1], label="Date index")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("PC1 vs PC2 score (color = time)"); axes[1].grid(alpha=0.3)

# Scree plot
axes[2].bar(range(1, len(evr)+1), evr*100, color="steelblue", alpha=0.8)
axes[2].set_xlabel("PC"); axes[2].set_ylabel("Variance explained (%)")
axes[2].set_title("Scree plot"); axes[2].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../figures/nb04_transport_pca.png", dpi=100)
plt.show()


Transport PCA — explained variance ratio:
  PC1: 0.9324  (93.2%)
  PC2: 0.0565  (5.6%)
  PC3: 0.0027  (0.3%)
  Cumulative: 99.2%


## 5. Variance Decomposition

How much of the total cross-sectional distributional variation is factor-driven vs idiosyncratic?


In [7]:
vd = variance_decomposition_ot(panel, loadings_df)
print("Variance Decomposition in Wasserstein Space:")
print(f"  Total variation (mean W₂²) : {vd['total_variation']:.2e}")
print(f"  Factor explained           : {vd['factor_explained_fraction']:.4f}  ({vd['factor_explained_fraction']*100:.1f}%)")
print(f"  Idiosyncratic              : {vd['idiosyncratic_fraction']:.4f}  ({vd['idiosyncratic_fraction']*100:.1f}%)")
print("Done.")


Variance Decomposition in Wasserstein Space:
  Total variation (mean W₂²) : 3.46e-05
  Factor explained           : 0.0143  (1.4%)
  Idiosyncratic              : 0.9857  (98.6%)
Done.
